# Optical-pumping resonance — D1 pump frequency from F=4 blowaway loss

For runs of `experiments/optical_pump_resonance_scan.py` (`*-OpticalPumpResoScan.h5`).

With the repumper off, a **resonant** D1 pump drives atoms out of F=3; whatever decays into F=4 stays there, and the F=4 blowaway ejects it. So

- retention **low**  → pump was resonant (atoms reached F=4, blown away)
- retention **high** → pump was off resonance (atoms stayed in F=3)

and the dip centre is the value for `f_pump_D1`.

- **A** — retention vs pump frequency, Wilson error bars, Gaussian-dip fit
- **B** — shot-1 / shot-2 count histograms against the threshold: is the atom call clean?
- **C** — loading and retention vs scan repeat: did anything drift?

## Check contrast before you believe any centre

This method is brutally sensitive — **one** scattering event that lands in F=4 loses the atom. So a strong or long pump removes atoms even far off resonance, retention goes low *everywhere*, and there is no dip to centre because the whole window is inside the resonance.

`summarize()` therefore reports **edge retention** (the most off-resonant points available) and **contrast** *before* fitting, and **suppresses the fit** when the edges are already depleted rather than returning a confident-looking number fitted to noise.

Mind the scale too: the scanned AOM frequency is not the optical detuning (a double-passed AOM doubles the step), and the Cs D1 natural linewidth is ~4.6 MHz. A ±2 MHz AOM scan covers only about one linewidth, so a saturated line is easily broader than the entire scan.

**Threshold is not taken on trust** — the kernel decided loaded/retained with whatever `atom_threshold` was live, but the raw counts are stored, so the call is remade here. `THRESHOLD = None` uses the archived value; `"stored"` uses the kernel's booleans.

The fit **pins the baseline** to the measured off-resonant plateau instead of fitting it (a free baseline is degenerate against a broad edge dip) and solves the amplitude in closed form. No scipy, no `np.linalg` — the BLAS in `artiq7_claude` segfaults.

All logic is imported from `analyze_op_resonance.py`; this notebook keeps no copy.

In [ ]:
# ---------------------------------------------------------------------------
# KERNEL CHECK -- run this FIRST.
# matplotlib cannot draw in the `artiq7_claude` env (numpy 1.23.5, broken BLAS:
# its affine transforms call np.dot while rendering, so plotting SEGFAULTS and
# the client reports "The Kernel crashed"). This analysis needs only
# numpy / h5py / matplotlib, NOT artiq -- so run it on plain Python 3.11:
#     VS Code -> kernel picker (top right) -> Python 3.11
# The probe runs np.dot in a SUBPROCESS, so a broken BLAS kills the child
# process instead of this kernel.
# ---------------------------------------------------------------------------
import subprocess, sys

_blas = subprocess.run(
    [sys.executable, "-c",
     "import numpy as np; np.dot(np.ones((8, 8)), np.ones((8, 8)))"],
    capture_output=True)
if _blas.returncode != 0:
    print("!" * 74)
    print("!! BROKEN BLAS in this kernel -- plotting WILL crash it.")
    print("!!   kernel : {}".format(sys.executable))
    print("!!   fix    : switch the kernel to Python 3.11, then re-run.")
    print("!" * 74)
else:
    print("BLAS ok -- plotting is safe in this kernel.")
    print("kernel:", sys.executable)

# Backend by what is importable -- never a bare `%matplotlib widget` (it needs
# ipympl; without it the kernel falls back to Qt, where plt.show() blocks).
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
except ImportError:
    get_ipython().run_line_magic("matplotlib", "inline")

import os, importlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

sys.path.insert(0, os.getcwd())
import analyze_op_resonance as A
importlib.reload(A)          # picks up edits to the .py without restarting
from analyze_op_resonance import (find_run, find_runs, load_run,
                                  retention_curve, contrast_report, fit_dip,
                                  wilson, summarize, analyze)

print("matplotlib backend:", matplotlib.get_backend())
print("runs on disk:", [A._rid_of(p) for p in find_runs()])

## Settings

In [ ]:
RID       = None   # None = latest run; or e.g. 2711
THRESHOLD = None   # counts on (shot - background). None = the run's archived
                   # value; a number overrides; "stored" = kernel's booleans.

## One run

In [ ]:
fig = analyze(find_run(RID), threshold=THRESHOLD)

## The retention table

Per-frequency counts, so you can see how much data sits behind each point.

In [ ]:
data, meta, rid = load_run(find_run(RID), threshold=THRESHOLD)
c = retention_curve(data)

print("  f_pump [MHz]   loaded  retained  retention  68% interval")
for i in np.argsort(c["f"]):
    bar = "#" * int(round(20 * (c["p"][i] if np.isfinite(c["p"][i]) else 0)))
    print("    %10.4f     %4d      %4d     %.3f    [%.3f, %.3f]  %s"
          % (c["f"][i] / 1e6, c["n_loaded"][i], c["n_retained"][i],
             c["p"][i], c["lo"][i], c["hi"][i], bar))

## Is the pump too strong? (contrast diagnostic)

If `edge` retention is already low, the pump depletes off resonance too and the scan has no contrast. Compare `edge` against what the atom survives with **no pump at all** — take a run with `OP_time_us = 0` as the null control and read its retention here.

In [ ]:
con = contrast_report(c)
print("pump used      : %.0f us at amp %.3f  (calibrated: %.0f us at %.3f)"
      % (1e6 * meta["t_pump_s"], meta["amp_d1"],
         1e6 * meta["t_optical_pump_live"], meta["amp_pump_D1_live"]))
print("edge retention : %.3f   minimum : %.3f   contrast : %.3f"
      % (con["edge"], con["min"], con["contrast"]))
print()
if con["depleted"]:
    print("=> SATURATED. Reduce OP_time_us and/or OP_amp until edge retention")
    print("   climbs back toward the no-pump survival, then re-scan.")
elif con["flat"]:
    print("=> FLAT. Contrast is within ~3x the error bars: more shots per")
    print("   point, or widen / recentre the scan.")
else:
    print("=> Usable contrast. The dip centre in panel A is meaningful.")

## Compare runs

Overlay runs at different pump strength/duration. The resonance **centre** should not move; only the depth and width should. A "dip" that shifts with pump power is a power-broadening artefact, not the line centre.

In [ ]:
RIDS = [A._rid_of(p) for p in find_runs()][-4:]    # last few runs; edit freely

plt.figure(figsize=(9.5, 5))
for rid in RIDS:
    r = summarize(find_run(rid), threshold=THRESHOLD, verbose=False)
    if r is None:
        continue
    cc, mm = r["curve"], r["meta"]
    plt.errorbar(cc["f"] / 1e6, cc["p"],
                 yerr=[np.maximum(0, cc["p"] - cc["lo"]),
                       np.maximum(0, cc["hi"] - cc["p"])],
                 fmt="-o", ms=3, lw=1, capsize=2, alpha=0.8,
                 label="RID %d  %.0f us @ %.3f"
                       % (rid, 1e6 * mm["t_pump_s"], mm["amp_d1"]))
    if r["fit"] is not None and r["contrast"] is not None \
            and not r["contrast"]["depleted"] and r["fit"]["significant"]:
        plt.axvline(r["fit"]["fc"] / 1e6, ls=":", lw=1, alpha=0.6)
plt.xlabel("D1 pump frequency (AOM) [MHz]")
plt.ylabel("retention (retained / loaded)")
plt.ylim(0, 1.05)
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()

## Reading the result — check before believing it

1. **Contrast first.** If the edges are depleted there is no centre to quote, whatever the grid minimum says. The fit is deliberately suppressed in that case.
2. **Panel B** — the two histograms should be bimodal with the threshold in the valley. If it cuts a peak, both `loaded` and `retained` are miscalled and the dip depth is wrong. Try a few `THRESHOLD` values and check the centre doesn't move.
3. **Panel C** — loading and retention should be flat across repeats; a trend smears the dip, since the scan is swept repeatedly.
4. **Scale** — a ±2 MHz AOM scan is only ~1 linewidth of optical detuning. If the curve is flat, widen the range before concluding the resonance isn't there.
5. **Saturated bottom** — points floored near zero retention give a flat-bottomed dip whose centre is only approximate; the summary says so.
6. **Cross-check with the repump control.** Re-run with `OP_repump_amp ≈ 0.38`: with the repumper restored, pumping returns atoms to F=3 and the dip should wash out. If it doesn't, the loss isn't coming from F=4 accumulation.
7. The experiment also stores its own on-worker fit in `opr_fit_resonance_Hz`. This notebook is independent — prefer it, because it reports contrast first and refuses to quote a centre the data can't support.